# Decision records and evidence-gated promotion

Every comparison in the [lifecycle](../docs/genai-lifecycle.md) ends in an
explicit decision — `adopt`, `reject`, or `inconclusive` — and only adopted
changes may move a production prompt alias. This credential-free lab walks the
complete loop with the shared SDK toolkit:

1. Score a baseline and a changed prompt with the same deterministic scorers
   used by release gates and production monitoring.
2. Apply a release gate with absolute thresholds and regression limits.
3. Record the outcome as a strict `DecisionRecord` — an `adopt` that cites a
   failing gate is rejected at the contract.
4. Watch `PromptManager.promote()` refuse to move the `production` alias
   without adopt-grade evidence.

The deterministic cells run with zero credentials. The final cell persists the
decision run and moves the alias against a connected workspace and stays off
by default. The practice-to-machinery map for all of this is the
[LLMOps playbook](../docs/llmops-playbook.md).

## 1. Score the baseline and the change with shared scorers

The fictional Aster Ridge earnings-summary assistant compares two exact prompt
templates: the baseline summarizes supplied facts, and the change adds exactly
one requirement — cite the supplied `source_id` once. `aai_core.scorers` holds
the deterministic scorer definitions, so this notebook, the CI release gate,
and production monitoring cannot drift apart. All figures are synthetic.

In [ ]:
from aai_core.scorers import score_all

BASELINE_TEMPLATE = (
    "Summarize only facts stated in {{earnings_excerpt}} for {{question}}."
)
CHANGE_TEMPLATE = (
    "Summarize only facts stated in {{earnings_excerpt}} for {{question}}. "
    "Cite {{source_id}} exactly once."
)

EVAL_CASES = [
    {
        "case_id": "quarterly-results",
        "source_id": "ARS-FY25-Q2-RESULTS",
        "expectations": {
            "expected_response": (
                "Fictional quarterly revenue reached $128.4 million with a "
                "12.1 percent operating margin [source: ARS-FY25-Q2-RESULTS]."
            )
        },
    },
    {
        "case_id": "forward-guidance",
        "source_id": "ARS-FY25-Q3-GUIDE",
        "expectations": {
            "expected_response": (
                "Fictional guidance projects $134 million revenue and a "
                "12.8 percent margin [source: ARS-FY25-Q3-GUIDE]."
            )
        },
    },
    {
        "case_id": "cash-and-risk",
        "source_id": "ARS-FY25-Q2-CASH-RISK",
        "expectations": {
            "expected_response": (
                "Fictional free cash flow was $21.7 million while inventory "
                "grew 18 percent [source: ARS-FY25-Q2-CASH-RISK]."
            )
        },
    },
]

BASELINE_ANSWERS = {
    "quarterly-results": (
        "Fictional quarterly revenue reached $128.4 million with a "
        "12.1 percent operating margin."
    ),
    "forward-guidance": (
        "Fictional guidance projects $134 million revenue and a "
        "12.8 percent margin."
    ),
    "cash-and-risk": (
        "Fictional free cash flow was $21.7 million while inventory "
        "grew 18 percent."
    ),
}
CHANGE_ANSWERS = {
    case["case_id"]: expected
    for case, expected in (
        (case, case["expectations"]["expected_response"]) for case in EVAL_CASES
    )
}


def citation_rate(answers):
    cited = sum(
        1
        for case in EVAL_CASES
        if answers[case["case_id"]].count(case["source_id"]) == 1
    )
    return cited / len(EVAL_CASES)


def mean_metrics(answers):
    totals = {}
    for case in EVAL_CASES:
        for name, value in score_all(
            answers[case["case_id"]], case["expectations"]
        ).items():
            totals.setdefault(name, []).append(value)
    metrics = {name: sum(values) / len(values) for name, values in totals.items()}
    metrics["citation_rate"] = citation_rate(answers)
    return metrics


baseline_metrics = mean_metrics(BASELINE_ANSWERS)
change_metrics = mean_metrics(CHANGE_ANSWERS)
for name in sorted(change_metrics):
    print(f"{name}: baseline={baseline_metrics[name]:.2f} "
          f"change={change_metrics[name]:.2f}")

## 2. Apply the deterministic release gate

`GatePolicy` states the release requirements once: the citation requirement is
absolute, and quality must not regress against the accepted baseline by more
than the stated budget. The gate consumes plain metric mappings here; against
a connected workspace the same `apply_gate()` consumes the native
`mlflow.genai.evaluate()` result unchanged.

In [ ]:
from aai_core.evaluation import (
    GatePolicy,
    MetricDirection,
    MetricRule,
    apply_gate,
)

gate_policy = GatePolicy(
    rules=(
        MetricRule(
            metric="citation_rate",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
        MetricRule(
            metric="keyword_coverage",
            direction=MetricDirection.HIGHER,
            required=0.8,
            max_regression=0.05,
        ),
        MetricRule(
            metric="response_length_ok",
            direction=MetricDirection.HIGHER,
            required=1.0,
        ),
    )
)

change_gate = apply_gate(
    change_metrics, policy=gate_policy, baseline_metrics=baseline_metrics
)
print(f"change gate passed: {change_gate.passed}")

failing_gate = apply_gate(
    baseline_metrics, policy=gate_policy, baseline_metrics=baseline_metrics
)
for failure in failing_gate.failures:
    print(f"baseline would fail on {failure.metric}: {failure.reason}")

## 3. Record an explicit decision

A decision is persisted evidence, not a comment. `DecisionRecord` binds the
vocabulary to what it was decided from: the baseline and change runs, the gate
result, the exact template content digest, and the release digest. The
contract refuses contradictory evidence — an `adopt` cannot cite a failing
gate — and refuses personal identity in `decided_by`.

In [ ]:
from pydantic import ValidationError

from aai_core.decisions import Decision, DecisionRecord
from aai_core.prompts import prompt_digest

try:
    DecisionRecord(
        decision=Decision.ADOPT,
        change_id="earnings-summary-prompt-v2",
        change_summary="Require one exact source citation.",
        rationale="This adopt contradicts its own gate evidence.",
        gate=failing_gate,
    )
except ValidationError as error:
    print(f"contradiction refused: {error.errors()[0]['msg']}")

adopt_record = DecisionRecord(
    decision=Decision.ADOPT,
    change_id="earnings-summary-prompt-v2",
    change_summary="Require one exact source citation.",
    rationale=(
        "Citation rate reached 1.0 on every case with no keyword-coverage "
        "regression against the accepted baseline."
    ),
    gate=change_gate,
    prompt_digest=prompt_digest(CHANGE_TEMPLATE),
    decided_by="group:app-owners",
)
# The connected cell below binds baseline_run_id/change_run_id to real runs.
print("searchable run tags:", adopt_record.as_tags())

## 4. Promotion refuses to move without adopt-grade, version-bound evidence

Aliases are deployment pointers, never release evidence. `promote()` checks
the evidence before it touches any registry, so both refusals below run with
zero credentials: a failing gate or a `reject`/`inconclusive` decision is
refused outright, and even a passing gate is refused when it is not bound to
template content. At promotion time the registry version's actual template
is digested and compared against the bound digest, so evidence gathered for
one template can never move the alias onto another version.

In [ ]:
from aai_core.decisions import Decision, DecisionRecord
from aai_core.prompts import PromptManager, PromptPromotionError, prompt_digest
from aai_core.testing import dev_settings

prompts = PromptManager(
    context=dev_settings().resource, catalog="main", schema="app"
)
print("change template digest:", prompt_digest(CHANGE_TEMPLATE)[:16], "...")

reject_record = DecisionRecord(
    decision=Decision.REJECT,
    change_id="earnings-summary-prompt-v2",
    change_summary="Require one exact source citation.",
    rationale="The gate failed the absolute citation requirement.",
    gate=failing_gate,
)
try:
    prompts.promote("earnings_summary", version=2, evidence=reject_record)
except PromptPromotionError as error:
    print(f"[{error.code}] {error}")

# A passing gate alone is not enough: it names no template content, so
# promotion refuses it until a digest binds it to the exact version.
try:
    prompts.promote("earnings_summary", version=2, evidence=change_gate)
except PromptPromotionError as error:
    print(f"[{error.code}] {error}")

## 5. Persist the decision and move the alias (connected)

Flip the flag after completing the connected setup lab
(`05_connected_setup.ipynb`). The cell registers the changed template
idempotently by content digest, writes the decision as a governed run with
searchable `aai.decision` tags and a `decision.json` artifact, and only then
moves the `production` alias — citing the same adopt evidence. Rerunning it
never mints duplicate prompt versions.

Graduation map: `prompt-app` templates carry this loop as
`scripts/register_prompt.py`, `evals/evaluate.py`, and
`scripts/promote_prompt.py`; the decision run makes the outcome searchable
next to the evaluation evidence.

In [ ]:
PERSIST_EVIDENCE_TO_DATABRICKS = False

if PERSIST_EVIDENCE_TO_DATABRICKS:
    from aai_core import bootstrap
    from aai_core.decisions import record_decision

    context = bootstrap()
    registered = context.prompts.ensure_version(
        "earnings_summary",
        CHANGE_TEMPLATE,
        commit_message="Require one exact source citation",
        tags={"experiment_role": "change"},
    )
    decision_run_id = record_decision(
        adopt_record, experiments=context.experiments
    )
    context.prompts.promote(
        "earnings_summary",
        version=registered.version,
        evidence=adopt_record,
    )
    print(
        f"decision run {decision_run_id} recorded; production alias moved "
        f"to version {registered.version}"
    )